# 04 · BiLSTM + attention (trained from scratch)

**Input:** `train/val/test.pkl`, `class_weights.json`. **Output:** `outputs/bilstm/` and `outputs/predictions/bilstm_attention.*`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from IPython.display import display
from tqdm.auto import tqdm

from src.config import BILSTM_DIR, CFG, DEVICE, SEED, set_seed, setup_notebook
from src.data import Vocab, collate_lstm, load_class_weights, load_splits, make_loader
from src.evaluation import plot_confusion, report, save_predictions
from src.explain import show_attention
from src.models import BiLSTMAttention, count_params
from src.text import tokenize
from src.training import fit_bilstm, predict_lstm

setup_notebook()

RETRAIN = False

CKPT_PATH, VOCAB_PATH = BILSTM_DIR / 'model.pt', BILSTM_DIR / 'vocab.json'
HIST_PATH, INFO_PATH = BILSTM_DIR / 'history.csv', BILSTM_DIR / 'train_info.json'
TRAIN_MODEL = RETRAIN or not CKPT_PATH.exists()
print('Training the BiLSTM from scratch' if TRAIN_MODEL else f'Checkpoint found -> loading from {CKPT_PATH}')

train_df, val_df, test_df = load_splits()
y_train, y_val, y_test = train_df['label'].values, val_df['label'].values, test_df['label'].values
CLASS_WEIGHTS_T = torch.tensor(load_class_weights(), device=DEVICE)
print(f'train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}')

## 1. Architecture
`Embedding (V×128) → Dropout → BiLSTM (128 per direction) → Attention pooling → Dropout → Linear (256→3)`

## 2. Tokenisation and vocabulary

In [ ]:
print(tokenize("Not as good as I expected... it's OK, but I won't buy it again!!"))

t0 = time.time()
train_tokens = [tokenize(t) for t in tqdm(train_df['text'], desc='Tokenising train', leave=False)]
val_tokens = [tokenize(t) for t in val_df['text']]
test_tokens = [tokenize(t) for t in test_df['text']]

if TRAIN_MODEL:
    vocab = Vocab.build(train_tokens, CFG['lstm_max_vocab'], CFG['lstm_min_freq'])
    vocab.save(VOCAB_PATH)
    counter = vocab.counter
else:
    vocab = Vocab.load(VOCAB_PATH)
    from collections import Counter
    counter = Counter(tok for toks in train_tokens for tok in toks)

MAX_LEN = CFG['lstm_max_len']
train_ids = [vocab.encode(t, MAX_LEN) for t in train_tokens]
val_ids = [vocab.encode(t, MAX_LEN) for t in val_tokens]
test_ids = [vocab.encode(t, MAX_LEN) for t in test_tokens]

covered = sum(c for w, c in counter.items() if w in vocab.stoi)
train_lens = np.array([len(t) for t in train_tokens])
print(f'Distinct words in train: {len(counter):,} | vocabulary size: {len(vocab):,}')
print(f'Share of train tokens in the vocabulary: {covered / sum(counter.values()):.2%}')
print(f'Share of reviews truncated (> {MAX_LEN} tokens): {(train_lens > MAX_LEN).mean():.2%}')
print(f'Time: {time.time() - t0:.0f}s')
print('Encoding example:', train_tokens[0][:12], '->', train_ids[0][:12])
del train_tokens, val_tokens, test_tokens, counter

## 3. DataLoader with length bucketing

In [ ]:
BS = CFG['lstm_batch_size']
train_loader = make_loader(train_ids, y_train, BS, shuffle=True, collate_fn=collate_lstm)
val_loader = make_loader(val_ids, y_val, BS, shuffle=False, collate_fn=collate_lstm)
test_loader = make_loader(test_ids, y_test, BS, shuffle=False, collate_fn=collate_lstm)
_, xb, lb, yb = next(iter(train_loader))
print(f'Batches per epoch: {len(train_loader):,} | one batch: x={tuple(xb.shape)}, lengths {lb.min().item()}–{lb.max().item()}')

## 4. Model

In [ ]:
set_seed(SEED)
lstm_model = BiLSTMAttention(len(vocab), CFG['lstm_emb_dim'], CFG['lstm_hidden'], CFG['lstm_layers'],
                             3, CFG['lstm_dropout']).to(DEVICE)
print(lstm_model)
n_params = count_params(lstm_model)
print(f'Trainable parameters: {n_params:,}')
pd.Series({name: p.numel() for name, p in lstm_model.named_parameters()}, name='parameters').to_frame()

## 5. Training

In [ ]:
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_T)

if TRAIN_MODEL:
    history, best_f1, train_time = fit_bilstm(
        lstm_model, train_loader, val_loader, y_val, criterion,
        epochs=CFG['lstm_epochs'], lr=CFG['lstm_lr'], patience=CFG['lstm_patience'], ckpt_path=CKPT_PATH)
    hist = pd.DataFrame(history)
    hist.to_csv(HIST_PATH, index=False)
    INFO_PATH.write_text(json.dumps({'train_time_s': train_time, 'best_val_macro_f1': best_f1}, indent=2))
else:
    lstm_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    hist = pd.read_csv(HIST_PATH) if HIST_PATH.exists() else None
    info = json.loads(INFO_PATH.read_text()) if INFO_PATH.exists() else {}
    train_time, best_f1 = info.get('train_time_s'), info.get('best_val_macro_f1', float('nan'))
    print(f'Loaded {CKPT_PATH}')
print(f'Training time: {(train_time or 0) / 60:.1f} min | best validation Macro-F1: {best_f1:.4f}')

In [ ]:
if hist is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(hist['epoch'], hist['train_loss'], 'o-', label='train')
    axes[0].plot(hist['epoch'], hist['val_loss'], 's-', label='validation')
    axes[0].set(title='BiLSTM – loss (class-weighted cross-entropy)', xlabel='Epoch', ylabel='Loss')
    axes[0].legend()
    axes[1].plot(hist['epoch'], hist['val_acc'], 'o-', label='val accuracy')
    axes[1].plot(hist['epoch'], hist['val_macro_f1'], 's-', label='val Macro-F1')
    axes[1].set(title='BiLSTM – validation metrics', xlabel='Epoch')
    axes[1].legend()
    for ax in axes:
        ax.xaxis.get_major_locator().set_params(integer=True)
    plt.tight_layout()
    plt.show()
    display(hist.round(4))

## 6. Test-set evaluation

In [ ]:
t0 = time.time()
test_probs, _ = predict_lstm(lstm_model, test_loader)
infer_time = time.time() - t0
test_pred = test_probs.argmax(1)
report('BiLSTM + Attention', y_test, test_pred)
plot_confusion(y_test, test_pred, 'BiLSTM + Attention – confusion matrix (test)')
plt.show()

save_predictions('BiLSTM + Attention', test_probs,
                 {'train_time_s': train_time, 'infer_time_s': infer_time, 'n_params': n_params})

## 7. Interpretation: which words does the attention focus on?

In [ ]:
short = test_df[test_df['n_words'].between(15, 45)]
for lab in [0, 1, 2]:
    cand = short[(short['label'] == lab) & (test_pred[short.index] == lab)]
    if len(cand):
        show_attention(lstm_model, vocab, cand.iloc[0]['text'], MAX_LEN, lab)
show_attention(lstm_model, vocab, "I really wanted to like this coffee, but it tastes burnt and I will not buy it again.", MAX_LEN)